In [ ]:
from config_dirs import INPUT_1_2_PMIDS, OUTPUT_1_2_OPENAI_INPUT
import pandas as pd
from Bio import Entrez
import time
import re

# Configuration 
ROOT_DIR = "/path/to/INNOVA_FINAL"  
INPUT_FILE = OUTPUT_1_1_PMIDS
OUTPUT_FILE = f"{ROOT_DIR}/1.1.Data_retrival/exact_matched_rows_with_keywords.xlsx"

Entrez.email = "your_email@example.com"  

# Function to retrieve PubMed data
def fetch_pubmed_data(pmids):
    records = []
    for start in range(0, len(pmids), 200):  # batch of 200 PMIDs   
        batch_pmids = pmids[start:start+200]
        handle = Entrez.efetch(db="pubmed", id=",".join(batch_pmids), rettype="xml")
        results = Entrez.read(handle)
        handle.close()
        for article in results['PubmedArticle']:
            try:
                medline = article['MedlineCitation']
                article_info = medline['Article']
                pmid = medline['PMID']
                title = article_info.get('ArticleTitle', '')
                abstract = ' '.join([t for t in article_info.get('Abstract', {}).get('AbstractText', [])])
                journal = article_info.get('Journal', {}).get('Title', '')
                pub_date = article_info.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {}).get('Year', '')
                authors = '; '.join([f"{a.get('ForeName','')} {a.get('LastName','')}" 
                                     for a in article_info.get('AuthorList', []) if 'LastName' in a])
                doi = ''
                for id_elem in article_info.get('ELocationID', []):
                    if id_elem.attributes.get('EIdType') == 'doi':
                        doi = str(id_elem)
                mesh_terms = [m['DescriptorName'] for m in medline.get('MeshHeadingList', [])] if 'MeshHeadingList' in medline else []
                pub_types = [pt for pt in article_info.get('PublicationTypeList', [])]

                # Normalize MeSH terms
                mesh_normalized = [re.sub(r'[^a-z0-9]+', ' ', mt.lower()) for mt in mesh_terms]

                records.append({
                    'PMID': str(pmid),
                    'ArticleTitle': title,
                    'AbstractText': abstract,
                    'Authors': authors,
                    'JournalTitle': journal,
                    'PublicationDate': pub_date,
                    'DOI': doi,
                    'MeSHTerms': '; '.join(mesh_terms),
                    'PublicationTypes': '; '.join(pub_types),
                    'MeSHTerms_normalized': ' '.join(mesh_normalized),
                    'Matched_Keywords': ''
                })
            except Exception as e:
                print(f"Errore con PMID {pmid}: {e}")
        time.sleep(0.3)  
    return records

# Load input
df_input = pd.read_csv(INPUT_FILE)
pmid_list = df_input['PMID'].astype(str).tolist()

# Fetch PubMed data
pubmed_records = fetch_pubmed_data(pmid_list)
df_pubmed = pd.DataFrame(pubmed_records)

# Filter for GWAS/WES/WGS 
keywords_pattern = re.compile(r'(genome[- ]wide association|GWAS|whole[- ]exome sequencing|WES|whole[- ]genome sequencing|WGS)', re.IGNORECASE)
df_pubmed['Matched_Keywords'] = df_pubmed['AbstractText'] + ' ' + df_pubmed['MeSHTerms_normalized']
df_filtered = df_pubmed[df_pubmed['Matched_Keywords'].str.contains(keywords_pattern)]

# Save output
df_filtered.to_csv(OUTPUT_FILE, index=False)
print(f"File filtrato salvato in {OUTPUT_FILE}, righe totali: {len(df_filtered)}")


In [ ]:
# Import necessary library
import pandas as pd

# Load the Excel file
file_path =  OUTPUT_FILE  
data = pd.read_excel(file_path)

# Display the first few rows to ensure the file loaded correctly
data.head()


,PMID,ArticleTitle,AbstractText,Authors,JournalTitle,PublicationDate,DOI,MeSHTerms,PublicationTypes,CitationCount,MeSHTerms_normalized,Matched_Keywords
0,18675980,Novel genetic variants linked to coronary arte...,BACKGROUND: It is uncertain whether the novel ...,M S Cunnington; B M Mayosi; D H Hall; P J Aver...,Atherosclerosis,2009,10.1016/j.atherosclerosis.2008.06.025,Adult; Aged; Carotid Arteries; Coronary Artery...,"Letter; Research Support, Non-U.S. Gov't",NaN,adult age carotid arteri coronary artery di...,adult age carotid arteri coronary artery di...
1,18523456,Replication of the Wellcome Trust genome-wide ...,Essential hypertension is a principal cardiova...,Georg B Ehret; Alanna C Morrison; Ashley A O'C...,European journal of human genetics : EJHG,2008,10.1038/ejhg.2008.102,"Adolescent; Adult; Aged; Aged, 80 and over; Bl...","Journal Article; Research Support, N.I.H., Ext...",NaN,"adolesc adult age aged, 80 and ov blood pr...","adolesc adult age aged, 80 and ov blood pr..."
2,20559952,Personalized medicine: a reality within this d...,Personalized medicine is defined as individual...,Robert Roberts,Journal of cardiovascular translational research,2008,10.1007/s12265-007-9001-1,Coronary Artery Disease; Genetic Predispositio...,Journal Article; Review,NaN,coronary artery diseas genetic predisposition...,coronary artery diseas genetic predisposition...
3,20206874,Genome-wide association studies in atherothrom...,Atherothrombotic diseases are complex diseases...,Luca Andrea Lotta,European journal of internal medicine,2010,10.1016/j.ejim.2009.11.003,Atherosclerosis; Genes; Genetic Linkage; Genet...,Journal Article; Review,NaN,atherosclerosi gene genetic linkag genetic ...,atherosclerosi gene genetic linkag genetic ...
4,20047492,Use of wrapper algorithms coupled with a rando...,Modern large-scale genetic association studies...,Andrei S Rodin; Anatoliy Litvinenko; Kathy Klo...,Journal of computational biology : a journal o...,2009,10.1089/cmb.2008.0037,Black or African American; Algorithms; Apolipo...,"Journal Article; Research Support, N.I.H., Ext...",NaN,black or african american algorithm apolipop...,black or african american algorithm apolipop...


In [2]:
# Count rows where 'AbstractText' is NA, 0, or '.'
invalid_rows_count = data[data['AbstractText'].isna() | (data['AbstractText'] == 0) | (data['AbstractText'] == '.')].shape[0]

# Display the count
invalid_rows_count


105

In [3]:
# Remove rows where 'AbstractText' is NA, 0, or '.'
filtered_data = data[~(data['AbstractText'].isna() | (data['AbstractText'] == 0) | (data['AbstractText'] == '.'))]

# Display the number of rows after removal
filtered_data.shape[0]


1452

In [4]:
# Save the filtered dataframe to an Excel file
filtered_data.to_excel("clean_first_step.xlsx", index=False)

print("Filtered dataframe saved as 'clean_first_step.xlsx'")


Filtered dataframe saved as 'clean_first_step.xlsx'


In [5]:
import pandas as pd

# 1. Read the Excel file
df = pd.read_excel("clean_first_step.xlsx")

# 2. Create the new column counting all characters in 'AbstractText'
#    astype(str) ensures non-string entries won't cause errors
df['abstract_letter_count'] = df['AbstractText'].astype(str).apply(len)

# 3. Reorder columns so 'abstract_letter_count' is immediately after 'AbstractText'
cols = df.columns.tolist()
abstract_idx = cols.index('AbstractText')  # Find the position of 'AbstractText'
# Remove the new column from the end
cols.remove('abstract_letter_count')
# Insert the new column right after 'AbstractText'
cols.insert(abstract_idx + 1, 'abstract_letter_count')
# Reassign the DataFrame to have columns in the new order
df = df[cols]

# 4. Save the updated DataFrame
df.to_excel("clean_first_step_abs_count.xlsx", index=False)

print("Updated DataFrame saved as 'clean_first_step_abs_count.xlsx'")


Updated DataFrame saved as 'clean_first_step_abs_count.xlsx'


In [6]:
# Select 10 random rows from the dataframe
random_rows = df.sample(n=10, random_state=42)  # Set random_state for reproducibility

# Save the random rows to an Excel file
random_rows.to_excel("test.xlsx", index=False)

print("10 random rows saved as 'test.xlsx'")


10 random rows saved as 'test.xlsx'


In [1]:
# Import necessary libraries
import pandas as pd

# Specify the file name
file_name = "clean_first_step_abs_count.xlsx"

# Load the Excel file into a pandas dataframe
df = pd.read_excel(file_name)

# Display the first few rows of the dataframe
df.head()


,PMID,ArticleTitle,AbstractText,abstract_letter_count,Authors,JournalTitle,PublicationDate,DOI,MeSHTerms,PublicationTypes,CitationCount,MeSHTerms_normalized,Matched_Keywords
0,30735646,Unraveling Difficult Answers: From Genotype to...,Genome-wide association studies (GWASs) have r...,347,Tarek Magdy; Paul W Burridge,Cell stem cell,2019,10.1016/j.stem.2019.01.008,"Cardiovascular Diseases; Chromosomes, Human, P...",Journal Article; Comment,NaN,"cardiovascular diseas chromosomes, human, pai...","cardiovascular diseas chromosomes, human, pai..."
1,28262776,Hypertension: From GWAS to functional genomics...,Several recent studies have provided insights ...,349,David L Mattson; Mingyu Liang,Nature reviews. Nephrology,2017,10.1038/nrneph.2017.21,Genome-Wide Association Study; Genomics; Human...,"Journal Article; Research Support, N.I.H., Ext...",NaN,genome-wide association studi genom human h...,genome-wide association studi genom human h...
2,21923062,Genomics in multiple myeloma research.,Multiple myeloma (MM) is the second most commo...,409,S Sevcíková; P Nemec; L Pour; R Hájek,Klinicka onkologie : casopis Ceske a Slovenske...,2011,NaN,Gene Expression Profiling; Genome-Wide Associa...,"Journal Article; Research Support, Non-U.S. Gov't",NaN,gene expression profil genome-wide associatio...,gene expression profil genome-wide associatio...
3,19912223,Deciphering the molecular basis of venous thro...,Venous thromboembolism (VTE) is a frequent dis...,432,Pierre-Emmanuel Morange; David-Alexandre Tregouet,British journal of haematology,2010,10.1111/j.1365-2141.2009.07975.x,Genetic Loci; Genetic Predisposition to Diseas...,Journal Article; Review,NaN,genetic loci genetic predisposition to diseas...,genetic loci genetic predisposition to diseas...
4,19912223,Deciphering the molecular basis of venous thro...,Venous thromboembolism (VTE) is a frequent dis...,432,Pierre-Emmanuel Morange; David-Alexandre Tregouet,British journal of haematology,2010,10.1111/j.1365-2141.2009.07975.x,Genetic Loci; Genetic Predisposition to Diseas...,Journal Article; Review,NaN,genetic loci genetic predisposition to diseas...,genetic loci genetic predisposition to diseas...


In [2]:
# Check the initial dimensions of the dataframe
initial_dimensions = df.shape
print(f"Initial dimensions: {initial_dimensions}")

Initial dimensions: (1448, 13)


In [3]:
# Drop duplicate rows and reset the index
df_filtered = df.drop_duplicates().reset_index(drop=True)

# Check the dimensions after removing duplicates
filtered_dimensions = df_filtered.shape
print(f"Dimensions after removing duplicates: {filtered_dimensions}")

Dimensions after removing duplicates: (1348, 13)


In [4]:
# Count the number of unique values in the PMID column
unique_pmids_count = df_filtered['PMID'].nunique()
print(f"Number of unique PMIDs: {unique_pmids_count}")


Number of unique PMIDs: 1348


In [ ]:
# Save the deduplicated dataframe as an Excel file
output_file_name = OUTPUT_1_2_OPENAI_INPUT
df_filtered.to_excel(output_file_name, index=False)

print(f"Dataframe saved as '{output_file_name}'")


Dataframe saved as 'OpenAI_Input.xlsx'
